# Net internal migration by age, by region (England & Wales)

Which ages are regions gaining or losing through moves within the UK? Tests two ideas from the age and migration analyses: London's "escalator" (young adults arrive, families and older workers leave) and rural regions losing young adults, which would help explain why they are older and churn less.

**Source:** ONS *Internal migration in England and Wales*, tables **IM{year}-T3a** (2012–2021 back series) and **IM{year}-T3** (2022–2024): inflow, outflow and net for each local authority by **five-year age group**, on 2023 LA boundaries (already in `raw/migration/`). Years ending June.

**Method:**
- LA → ITL2 2021 (crosswalk + spine), summing **net** moves. Moves between councils in the same region cancel out in the net, so the regional net is exactly its net exchange with the rest of the UK (including Scotland and NI).
- Pooled 2012–2024; **net migration rate** = net moves ÷ population in that age group (mid-year estimates, same years), % per year.
- **England & Wales only** (35 regions): the ONS age tables don't cover Scotland or NI regions.
- International migration is not included.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

BASE  = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
RAW   = BASE / "raw" / "migration"
DIR21 = BASE / "clean" / "itl2_2021"
FIG   = BASE / "analysis" / "demographics" / "charts"
sys.path.append(str(BASE / "analysis"))
from chart_setup import styles, alt, ECO, TXT, GREY

SOURCE = "Sources: ONS internal migration estimates (England and Wales); ONS mid-year population estimates; ONS Longitudinal Business Database; authors' calculations."
YEARS = range(2012, 2025)
BANDS = ["0-4", "5-9", "10-14", "15-19", "20-24", "25-29", "30-34", "35-39", "40-44", "45-49", "50-54", "55-59", "60-64", "65-69", "70-74", "75-79", "80-84", "85-89", "90+"]
remap = dict(zip(*[pd.read_csv(BASE / "lib" / "la_code_crosswalk.csv", dtype=str)[c] for c in ("old_code", "new_code")]))
spine = pd.read_csv(DIR21 / "geography_spine.csv"); la2itl = dict(zip(spine.la_code, spine.itl2_code))
names = spine.drop_duplicates("itl2_code").set_index("itl2_code").itl2_name

def read_t3(year):
    f, sheet = (RAW / "internalmigrationbackseries2012to2021.xlsx", f"IM{year}-T3a") if year <= 2021 else (RAW / f"internalmigration{year}tables.xlsx", f"IM{year}-T3")
    d = pd.read_excel(f, sheet_name=sheet, header=None)
    r = d.index[d[0].astype(str).str.strip() == "LA code"][0]
    d = d.iloc[r + 1:, :6]; d.columns = ["la_code", "la_name", "age", "inflow", "outflow", "net"]
    d = d.dropna(subset=["la_code"]); d = d[d.la_code.astype(str).str.match(r"^[EW]\d{8}$")]
    d[["inflow", "outflow", "net"]] = d[["inflow", "outflow", "net"]].apply(pd.to_numeric)
    return d.assign(year=year, age=d.age.astype(str).str.strip())

raw = pd.concat([read_t3(y) for y in YEARS], ignore_index=True)
raw["itl2_code"] = raw.la_code.replace(remap).map(la2itl)
assert raw.itl2_code.notna().all(), raw[raw.itl2_code.isna()].la_code.unique()
assert set(raw.age) == set(BANDS), set(raw.age) ^ set(BANDS)
net = raw.groupby(["itl2_code", "year", "age"]).net.sum().reset_index()

# check: summed over ages, net matches the region totals already built in build/migration.ipynb
tot = net.groupby(["itl2_code", "year"]).net.sum()
ref = pd.read_csv(DIR21 / "migration_itl2.csv").set_index(["itl2_code", "year"]).net_migration
gap = (tot - ref.reindex(tot.index)).abs()
print(f"{raw.la_code.nunique()} LAs -> {net.itl2_code.nunique()} ITL2 regions, {len(YEARS)} years. "
      f"Age-summed net vs region totals: median |gap| {gap.median():.0f}, max {gap.max():.0f} moves (ONS rounding)")

318 LAs -> 35 ITL2 regions, 13 years. Age-summed net vs region totals: median |gap| 0, max 2 moves (ONS rounding)


In [2]:
# population by the same five-year bands (pooled 2012-24) and net migration rates
sy = pd.read_csv(DIR21 / "age_single_year_itl2.csv").query("year >= 2012 and year <= 2024")
edges = list(range(0, 95, 5))
sy["age_band"] = pd.cut(sy.age.clip(upper=90), bins=edges + [200], right=False, labels=BANDS)
popb = sy.groupby(["itl2_code", "age_band"], observed=True).persons.sum().rename("pop").reset_index().rename(columns={"age_band": "age"})
popb["age"] = popb.age.astype(str)

LONDON = lambda s: s.str.startswith("TLI")
def rates(group_map):
    n = net.assign(g=net.itl2_code.map(group_map)).dropna(subset=["g"]).groupby(["g", "age"]).net.sum()
    p = popb.assign(g=popb.itl2_code.map(group_map)).dropna(subset=["g"]).groupby(["g", "age"])["pop"].sum()
    return (100 * n / p).rename("rate").reset_index()

# groups: London; the other 30 E&W regions in thirds by population-weighted density (2018-24 average)
pw = pd.read_csv(DIR21 / "pop_weighted_density_itl2.csv").query("year >= 2018")
pw = (pw.assign(w=pw.pop_weighted_density * pw.population).groupby("itl2_code")[["w", "population"]].sum().pipe(lambda g: g.w / g.population))
ew = pd.Series(sorted(net.itl2_code.unique()))
other = ew[~LONDON(ew)]
third = pd.qcut(pw.reindex(other), 3, labels=["Least dense third", "Middle third", "Densest third"]).astype(str)
GROUPS = {**{c: "London" for c in ew[LONDON(ew)]}, **third.to_dict()}
for g in ["Densest third", "Middle third", "Least dense third"]:
    print(f"{g}: " + ", ".join(names[c] for c, v in GROUPS.items() if v == g))
gr = rates(GROUPS)
print(gr.pivot(index="age", columns="g", values="rate").loc[BANDS].round(2).to_string())

Densest third: Northumberland, and Tyne and Wear, Greater Manchester, Merseyside, West Yorkshire, Leicestershire, Rutland and Northamptonshire, West Midlands, Bedfordshire and Hertfordshire, Surrey, East and West Sussex, Hampshire and Isle of Wight, Gloucestershire, Wiltshire and Bath/Bristol area
Middle third: Tees Valley and Durham, Lancashire, East Yorkshire and Northern Lincolnshire, South Yorkshire, Derbyshire and Nottinghamshire, Essex, Berkshire, Buckinghamshire and Oxfordshire, Kent, Devon, East Wales
Least dense third: Cumbria, Cheshire, North Yorkshire, Lincolnshire, Herefordshire, Worcestershire and Warwickshire, Shropshire and Staffordshire, East Anglia, Dorset and Somerset, Cornwall and Isles of Scilly, West Wales and The Valleys
g      Densest third  Least dense third  London  Middle third
age                                                          
0-4             0.31               1.07   -3.15          0.77
5-9             0.05               0.77   -2.07          0.50

In [3]:
# chart 1: net migration rate by age, London vs density groups (lines)
GORD = ["London", "Densest third", "Middle third", "Least dense third"]
GCOL = [ECO["red"], ECO["blue-dark"], ECO["blue-light"], ECO["orange"]]
gr["pos"] = gr.age.map({b: i for i, b in enumerate(BANDS)})
LBL = {b: b.replace("-", "–") for b in BANDS}
gr["age_l"] = gr.age.map(LBL)
XA = alt.X("age_l:N", sort=[LBL[b] for b in BANDS], title="Age group", axis=alt.Axis(labelAngle=0, labelExpr="indexof(['0–4','10–14','20–24','30–34','40–44','50–54','60–64','70–74','80–84','90+'], datum.label) >= 0 ? datum.label : ''"))
end = gr[gr.age == "90+"]
lines = alt.layer(
    alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.5).encode(y="v:Q"),
    alt.Chart(gr).mark_line(strokeWidth=2.6, point=alt.OverlayMarkDef(size=28)).encode(
        x=XA, y=alt.Y("rate:Q", title="Net internal migration, % of age group a year",
                      axis=alt.Axis(labelExpr="datum.value > 0 ? '+' + datum.label + '%' : datum.label + '%'")),
        color=alt.Color("g:N", scale=alt.Scale(domain=GORD, range=GCOL), legend=alt.Legend(title=None, orient="bottom")),
        tooltip=[alt.Tooltip("g:N", title="Group"), alt.Tooltip("age_l:N", title="Age"), alt.Tooltip("rate:Q", format="+.2f", title="Net, % a year")]),
).properties(title=alt.Title("London gains young adults and loses everyone else; rural regions do the reverse",
    subtitle=["Net internal migration (moves in minus moves out, within the UK) as % of each age group's population a year, pooled 2012–24.",
              "England and Wales: London, then the other 30 ITL2 regions in thirds by population-weighted density. International migration excluded.", SOURCE]))
styles.save(lines, str(FIG), "migration_net_by_age", width=600, height=360)
lines.properties(width=600, height=360)

alt.LayerChart(...)

In [4]:
# chart 2: heatmap, every E&W region x age group, sorted by job reallocation (2012-24)
HY = (2018, 2024)   # heatmap period: ranking and migration both pooled over 2018-24
reg = pd.read_csv(DIR21 / "job_flows_itl2.csv").query("analysis_window and year >= @HY[0] and year <= @HY[1]")
gj = reg.groupby("itl2_code")[["total_jc", "total_jd", "emp_avg"]].sum(); realloc = 100 * (gj.total_jc + gj.total_jd) / gj.emp_avg
n18 = net[net.year.between(*HY)].groupby(["itl2_code", "age"]).net.sum()
p18 = (sy[sy.year.between(*HY)].groupby(["itl2_code", "age_band"], observed=True).persons.sum()
         .rename_axis(["itl2_code", "age"]).rename(lambda a: str(a), level=1))
hm = (100 * n18 / p18).rename("rate").reset_index().dropna(subset=["rate"])
hm = hm[hm.itl2_code.isin(ew)]                               # England & Wales only
hm["region"] = hm.itl2_code.map(names).str.replace(" - ", " ").str.replace(" and ", " & ")
hm["age_l"] = hm.age.map(LBL)
hm["realloc"] = hm.itl2_code.map(realloc)
order = list(hm.drop_duplicates("itl2_code").sort_values("realloc", ascending=False).region)
lim = 4.0
# diverging scale with extra dark ends: white at 0, house red/blue at ±2%, dark red/navy at the extremes
DARK_RED = "#8a1030"
HEAT_SCALE = alt.Scale(domain=[-6, -2, 0, 2, 6], range=[DARK_RED, ECO["red"], "#ffffff", ECO["blue-light"], ECO["blue-dark"]],
                       interpolate="rgb", clamp=True)
heat = alt.Chart(hm).mark_rect(stroke="white", strokeWidth=0.5).encode(
    x=alt.X("age_l:N", sort=[LBL[b] for b in BANDS], title="Age group", axis=alt.Axis(labelAngle=0, orient="top",
            labelExpr="indexof(['0–4','10–14','20–24','30–34','40–44','50–54','60–64','70–74','80–84','90+'], datum.label) >= 0 ? datum.label : ''")),
    y=alt.Y("region:N", sort=order, title=None, axis=alt.Axis(labelLimit=260, labelFontSize=10)),
    color=alt.Color("rate:Q", scale=HEAT_SCALE,
                    legend=alt.Legend(title=["Net migration,", "% a year"], orient="right", gradientLength=220,
                                      labelExpr="datum.value > 0 ? '+' + datum.label + '%' : datum.label + '%'")),
    tooltip=[alt.Tooltip("region:N"), alt.Tooltip("age_l:N", title="Age"), alt.Tooltip("rate:Q", format="+.2f", title="Net, % a year"),
             alt.Tooltip("realloc:Q", format=".1f", title="Reallocation 2018–24 (%)")],
).properties(width=520, height=640, title=alt.Title("Where each age group is moving: high-churn regions gain people in their twenties",
    subtitle=["Net internal migration as % of each age group's population a year, pooled 2018–24. Blue = gaining, red = losing (darkest shades = ±6% or more).",
              "England and Wales ITL2 regions, sorted from highest to lowest job reallocation rate (2018–24). International migration excluded.", SOURCE]))
styles.save(heat, str(FIG), "migration_net_by_age_heatmap", width=520, height=640)

# young-adult (20-29) net rate vs reallocation
ya = hm[hm.age.isin(["20-24", "25-29"])].merge(p18.rename("pop").reset_index(), on=["itl2_code", "age"])
ya = (ya.assign(n=ya.rate * ya["pop"] / 100).groupby("itl2_code")[["n", "pop"]].sum().pipe(lambda g: 100 * g.n / g["pop"]))
print(f"Corr(net migration rate at 20–29, reallocation): {np.corrcoef(ya, realloc.reindex(ya.index))[0, 1]:.2f} (35 regions); "
      f"excl. London {np.corrcoef(ya[~LONDON(pd.Series(ya.index, index=ya.index))], realloc.reindex(ya.index)[~LONDON(pd.Series(ya.index, index=ya.index))])[0, 1]:.2f}")
heat

Corr(net migration rate at 20–29, reallocation): 0.56 (35 regions); excl. London 0.09


alt.Chart(...)

In [5]:
# two heatmaps side by side: 2012-17 and 2018-23, same row order (ranked by job reallocation 2018-23)
HP = {"2012–17": (2012, 2017), "2018–23": (2018, 2023)}
jfa = pd.read_csv(DIR21 / "job_flows_itl2.csv").query("analysis_window")
def churn(yrs):
    g = jfa[jfa.year.between(*yrs)].groupby("itl2_code")[["total_jc", "total_jd", "emp_avg"]].sum()
    return 100 * (g.total_jc + g.total_jd) / g.emp_avg
def heat_data(yrs):
    n = net[net.year.between(*yrs)].groupby(["itl2_code", "age"]).net.sum()
    p_ = (sy[sy.year.between(*yrs)].groupby(["itl2_code", "age_band"], observed=True).persons.sum()
            .rename_axis(["itl2_code", "age"]).rename(lambda a: str(a), level=1))
    h = (100 * n / p_).rename("rate").reset_index().dropna(subset=["rate"])
    h = h[h.itl2_code.isin(ew)].copy()
    h["region"] = h.itl2_code.map(names).str.replace(" - ", " ").str.replace(" and ", " & ")
    h["age_l"] = h.age.map(LBL); h["realloc"] = h.itl2_code.map(churn(yrs))
    return h
hd = {lab: heat_data(y) for lab, y in HP.items()}
rank = churn(HP["2018–23"]).reindex(ew)
ORDER_R = list(hd["2018–23"].drop_duplicates("itl2_code").assign(r=lambda x: x.itl2_code.map(rank)).sort_values("r", ascending=False).region)

def heat_panel(lab, first):
    return alt.Chart(hd[lab]).mark_rect(stroke="white", strokeWidth=0.5).encode(
        x=alt.X("age_l:N", sort=[LBL[b] for b in BANDS], title=None, axis=alt.Axis(labelAngle=0, orient="top",
                labelExpr="indexof(['0–4','20–24','40–44','60–64','80–84'], datum.label) >= 0 ? datum.label : ''")),
        y=alt.Y("region:N", sort=ORDER_R, title=None, axis=alt.Axis(labels=first, ticks=first, labelLimit=260, labelFontSize=10) if first else None),
        color=alt.Color("rate:Q", scale=HEAT_SCALE,
                        legend=alt.Legend(title=["Net migration,", "% a year"], orient="right", gradientLength=220,
                                          labelExpr="datum.value > 0 ? '+' + datum.label + '%' : datum.label + '%'")),
        tooltip=[alt.Tooltip("region:N"), alt.Tooltip("age_l:N", title="Age"), alt.Tooltip("rate:Q", format="+.2f", title="Net, % a year"),
                 alt.Tooltip("realloc:Q", format=".1f", title=f"Reallocation {lab} (%)")],
    ).properties(width=300, height=640, title=alt.Title(lab, anchor="start", fontSize=13))

heat2 = alt.hconcat(heat_panel("2012–17", True), heat_panel("2018–23", False), spacing=10).resolve_scale(color="shared").properties(
    title=alt.Title("The age pattern of internal migration has held steady: London gains twenty-somethings and loses older groups",
    subtitle=["Net internal migration as % of each age group's population a year, pooled over each period. Blue = gaining, red = losing (darkest shades = ±6% or more).",
              "England and Wales ITL2 regions (age data not published for Scottish or NI regions), ranked by job reallocation rate 2018–23. International migration excluded.",
              SOURCE]))
styles.save(heat2, str(FIG), "migration_net_by_age_heatmap_periods", width=300, height=640)
for lab in HP:
    h = hd[lab]; ya = h[h.age.isin(["20-24", "25-29"])]
    print(lab, "London net at 20–24:", round(h[(h.age == "20-24") & h.itl2_code.str.startswith("TLI")].rate.mean(), 2))
heat2

2012–17 London net at 20–24: 4.62
2018–23 London net at 20–24: 5.3


alt.HConcatChart(...)